<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
def load_data(seed=42):
    X, y = fetch_openml("Fashion-MNIST", version=1, as_frame=False, return_X_y=True)
    y = y.astype(int)

    # normalização simples
    X = X / 255.0

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    return X_train, X_test, y_train, y_test

Não é estritamente necessário normalizar dados para modelos baseados em árvores, pois eles não dependem de distância. No entanto, foi feita uma normalização simples dividindo por 255 para padronizar os valores dos pixels.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [3]:
def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

In [11]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

**Solução**:

A função evaluate realiza as predições utilizando o modelo treinado e calcula a acurácia comparando com os valores reais.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [8]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

    acc = evaluate(model, X_test, y_test)
    return acc

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**
O pipeline organiza todo o fluxo do experimento: carrega os dados, treina o modelo escolhido e avalia seu desempenho. Isso garante estrutura e reprodutibilidade.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

In [12]:
def train_adaboost(X_train, y_train, seed=42):
    base = DecisionTreeClassifier(max_depth=1, random_state=seed)

    model = AdaBoostClassifier(
        estimator=base,
        n_estimators=50,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model

O Random Forest apresentou melhor desempenho inicial, com maior estabilidade e, em geral, maior acurácia em comparação ao AdaBoost. Além disso, apresentou métricas mais equilibradas entre precisão, recall e F1-score.

**Solução**:

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

In [ ]:
for seed in [42, 7]:
    acc_rf = run_pipeline("rf", seed=seed)
    acc_ab = run_pipeline("ab", seed=seed)

    print(f"Seed {seed} → RF: {acc_rf:.4f} | AB: {acc_ab:.4f}")

Os resultados mudam levemente ao utilizar seeds diferentes, pois há aleatoriedade no treinamento dos modelos. No entanto, com a mesma seed, os resultados permanecem iguais, garantindo a reprodutibilidade do experimento.

**Solução**:

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

print("RF treino:", accuracy_score(y_train, rf_model.predict(X_train)))
print("RF teste:", accuracy_score(y_test, rf_model.predict(X_test)))

print("AB treino:", accuracy_score(y_train, ab_model.predict(X_train)))
print("AB teste:", accuracy_score(y_test, ab_model.predict(X_test)))

Existe overfitting quando a acurácia no treino é significativamente maior do que no teste. Nesse experimento, isso pode ocorrer dependendo dos parâmetros utilizados. O AdaBoost tende a ser mais sensível a overfitting dependendo da configuração, enquanto o Random Forest geralmente apresenta maior robustez.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
results = []

X_train, X_test, y_train, y_test = load_data(seed=42)

for n in [10, 50, 100]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_acc = accuracy_score(y_test, rf.predict(X_test))

    ab = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=n,
        random_state=42
    )
    ab.fit(X_train, y_train)
    ab_acc = accuracy_score(y_test, ab.predict(X_test))

    results.append({"model": "RF", "n_estimators": n, "acc": rf_acc})
    results.append({"model": "AB", "n_estimators": n, "acc": ab_acc})

pd.DataFrame(results)

O desempenho melhora com o aumento de n_estimators até certo ponto, depois tende a estabilizar. O AdaBoost costuma ser mais sensível a mudanças de hiperparâmetros, enquanto o Random Forest apresenta maior estabilidade.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [ ]:
def full_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

print("RF:", full_metrics(rf_model, X_test, y_test))
print("AB:", full_metrics(ab_model, X_test, y_test))

A acurácia não é suficiente para avaliar completamente os modelos, pois pode esconder problemas como desbalanceamento de classes. Por isso, é importante analisar também precisão, recall e F1-score.

A reprodutibilidade foi garantida com o uso de random_state em todas as etapas do experimento. Isso permite repetir os resultados de forma consistente.

Dois possíveis problemas metodológicos são a dependência de uma única divisão treino/teste e a limitação na escolha de hiperparâmetros.

O pipeline implementado é confiável, pois segue boas práticas de organização, controle de aleatoriedade e avaliação. No entanto, poderia ser melhorado com técnicas como validação cruzada.